In [1]:
import os
import json
import time
import hashlib
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# ── LangChain & vector store ──────────────────────────────────────────────────
try:
    from langchain.text_splitter import (
        RecursiveCharacterTextSplitter,
        SentenceTransformersTokenTextSplitter,
    )
    from langchain_community.vectorstores import Chroma
    from langchain_community.embeddings import HuggingFaceEmbeddings
    from langchain.schema import Document
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False
    print("LangChain not installed. Run: pip install langchain langchain-community chromadb")

LangChain not installed. Run: pip install langchain langchain-community chromadb


In [3]:
!pip install langchain langchain-community chromadb

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------------------------------- -- 2.4/2.5 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 10.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.6 MB/s  0:00:00
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/21.9 MB 11.8 MB/s eta 0:00:02
   -------- ------------------------------- 4.7/21.9 MB 11.8 MB/s eta 0:00:02
   ------------ --------------------------- 7.1/21.9 MB 11.8 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf2crf 0.1.33 requires tensorflow-addons>=0.8.2, which is not installed.


In [4]:
# ── Sentence transformers (embeddings) ───────────────────────────────────────
try:
    from sentence_transformers import SentenceTransformer
    import torch
    SENTENCE_TRANSFORMERS = True
except ImportError:
    SENTENCE_TRANSFORMERS = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [5]:
# Configuration
CONFIG = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model": "gpt-4o-mini",               # swap for claude-3-haiku for Anthropic
    "chunk_sizes": [256, 512, 1024],
    "chunk_overlaps": [32, 64, 128],
    "top_k_retrieval": 5,
    "rerank_top_k": 3,
    "bm25_weight": 0.3,
    "dense_weight": 0.7,
    "cache_ttl_hours": 24,
}

In [6]:

# ─────────────────────────────────────────────────────────────────────────────
# 1. DOCUMENT LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def load_documents(pdf_dir: str = "./airline_policies/",
                   complaint_csv: str = "./twitter_complaints.csv") -> List[Dict]:
    """
    QUESTION 1 (Document Loading):
    Load two document types into a unified corpus:
      1. Airline policy PDFs → structured regulatory text
      2. Twitter complaint data → unstructured customer voice

    This combination gives your RAG app BOTH the official policy answer
    AND awareness of what customers are frustrated about — a compelling
    business narrative: 'Our chatbot knows the policies AND understands
    the real pain points.'
    """
    documents = []

    # Load PDFs
    try:
        import fitz  # pymupdf
        pdf_path = Path(pdf_dir)
        if pdf_path.exists():
            for pdf_file in pdf_path.glob("*.pdf"):
                doc = fitz.open(pdf_file)
                for page_num, page in enumerate(doc):
                    text = page.get_text()
                    if len(text.strip()) > 50:
                        documents.append({
                            "content": text,
                            "source": pdf_file.name,
                            "page": page_num + 1,
                            "doc_type": "policy_pdf",
                        })
                logger.info(f"Loaded {len(doc)} pages from {pdf_file.name}")
        else:
            logger.warning(f"PDF directory not found: {pdf_dir}")
    except ImportError:
        logger.warning("PyMuPDF not installed. Run: pip install pymupdf")

    # Load complaint CSV
    if Path(complaint_csv).exists():
        df = pd.read_csv(complaint_csv)
        text_col = next((c for c in ["text", "tweet", "complaint", "content"]
                         if c in df.columns), df.columns[0])
        for _, row in df.iterrows():
            text = str(row[text_col])
            if len(text.strip()) > 20:
                documents.append({
                    "content": text,
                    "source": "twitter_complaints",
                    "doc_type": "customer_complaint",
                    "airline": row.get("airline", "Unknown"),
                })
    else:
        # Use synthetic airline policy text for demonstration
        logger.info("No real documents found — using synthetic airline policy corpus")
        documents = _generate_synthetic_corpus()

    logger.info(f"Total documents loaded: {len(documents)}")
    return documents


def _generate_synthetic_corpus() -> List[Dict]:
    """Generate a realistic airline policy corpus for demonstration."""
    policies = [
        {
            "content": """BAGGAGE POLICY — IndiGo Airlines
            Carry-on baggage: Economy class passengers are allowed one cabin bag of maximum 7 kg
            with maximum dimensions of 55 cm × 35 cm × 25 cm. Business class passengers are
            allowed two cabin bags totalling 14 kg. Excess carry-on baggage will be charged at
            Rs. 500 per kg at the gate. Liquids, aerosols, and gels exceeding 100 ml are not
            permitted in cabin baggage as per DGCA regulations.
            Checked baggage: Economy Light includes no free checked baggage. Economy includes
            15 kg free checked baggage. Economy Flexi includes 30 kg free checked baggage.
            Excess checked baggage fee is Rs. 350 per kg for domestic flights.""",
            "source": "indigo_baggage_policy.pdf", "page": 1, "doc_type": "policy_pdf",
        },
        {
            "content": """REFUND AND CANCELLATION POLICY — IndiGo Airlines
            Cancellations made more than 24 hours before departure: Rs. 3,500 cancellation fee
            for Economy class. Cancellations within 2 hours of departure are non-refundable.
            No-show policy: Passengers who fail to check in will forfeit the full ticket amount.
            Refund processing time: 7-10 working days for credit card refunds. Cash bookings
            refunded within 20 working days. Refunds will be credited to the original payment method.""",
            "source": "indigo_refund_policy.pdf", "page": 1, "doc_type": "policy_pdf",
        },
        {
            "content": """FLIGHT DELAY COMPENSATION — IndiGo Airlines
            For delays of 2-4 hours, passengers are entitled to meal vouchers of Rs. 200.
            For delays exceeding 4 hours on domestic routes, passengers may claim refund or
            rebooking at no extra charge. For delays exceeding 6 hours on international routes,
            passengers are entitled to hotel accommodation if the delay occurs between 22:00 and 06:00.
            Compensation claims must be submitted within 30 days of the incident. DGCA Civil
            Aviation Requirements require airlines to inform passengers of delays immediately.""",
            "source": "indigo_delay_policy.pdf", "page": 1, "doc_type": "policy_pdf",
        },
        {
            "content": """FREQUENT FLYER PROGRAMME — IndiGo 6E Rewards
            Points earned: 6 points per Rs. 100 spent on base fare. Status tiers: Silver (5,000
            points), Gold (25,000 points), Platinum (75,000 points). Points validity: 36 months
            from date of earning. Points can be redeemed for flight upgrades, seat selection, and
            partner hotel stays. Partner airline points transfer available at 2:1 ratio with Air
            Arabia and GoAir.""",
            "source": "indigo_loyalty_policy.pdf", "page": 1, "doc_type": "policy_pdf",
        },
        {
            "content": "@IndiGo6E my flight was delayed by 3 hours and no one told us anything! "
                        "Horrible experience at BLR airport. #IndiGofail",
            "source": "twitter_complaints", "doc_type": "customer_complaint", "airline": "IndiGo",
        },
        {
            "content": "@IndiGo6E I was charged excess baggage fee at the gate but my bag was "
                        "within the 7kg limit according to the home scale. Completely unfair!",
            "source": "twitter_complaints", "doc_type": "customer_complaint", "airline": "IndiGo",
        },
    ]
    return policies


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# 2. CHUNKING STRATEGY COMPARISON
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class ChunkingResult:
    strategy: str
    chunk_size: int
    n_chunks: int
    avg_chunk_len: float
    min_chunk_len: int
    max_chunk_len: int
    retrieval_precision_at_k: float = 0.0
    latency_ms: float = 0.0


def compare_chunking_strategies(documents: List[Dict],
                                  test_query: str = "What is the baggage allowance?") -> pd.DataFrame:
    """
    QUESTION 2 (Chunking Strategy — ablation study almost never seen in portfolios):
    Compare three chunking approaches empirically:

    1. Fixed-size chunking: Simple, fast, but breaks sentences mid-context
       → Good for: large-scale deployment where latency matters
    2. Sentence-aware chunking: Respects sentence boundaries
       → Good for: documents with coherent paragraphs
    3. Semantic chunking: Groups sentences by embedding similarity
       → Best for: documents with topic shifts (e.g., mixed policy documents)

    Show empirically which produces better retrieval quality on YOUR corpus.
    This ablation study is what separates junior from senior ML engineers.
    """
    corpus_texts = [d["content"] for d in documents]
    results = []

    strategies = {
        "Fixed-256": {"chunk_size": 256, "chunk_overlap": 32},
        "Fixed-512": {"chunk_size": 512, "chunk_overlap": 64},
        "Fixed-1024": {"chunk_size": 1024, "chunk_overlap": 128},
    }

    for strategy_name, params in strategies.items():
        t0 = time.time()

        if LANGCHAIN_AVAILABLE:
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=params["chunk_size"],
                chunk_overlap=params["chunk_overlap"],
                separators=["\n\n", "\n", ". ", " ", ""],
            )
            chunks = splitter.create_documents(corpus_texts)
            chunk_texts = [c.page_content for c in chunks]
        else:
            # Simple fallback chunking
            chunk_texts = []
            for text in corpus_texts:
                size = params["chunk_size"]
                for i in range(0, len(text), size - params["chunk_overlap"]):
                    chunk_texts.append(text[i:i+size])

        latency = (time.time() - t0) * 1000

        if chunk_texts:
            results.append(ChunkingResult(
                strategy=strategy_name,
                chunk_size=params["chunk_size"],
                n_chunks=len(chunk_texts),
                avg_chunk_len=np.mean([len(c) for c in chunk_texts]),
                min_chunk_len=min(len(c) for c in chunk_texts),
                max_chunk_len=max(len(c) for c in chunk_texts),
                latency_ms=latency,
            ))

    df = pd.DataFrame([vars(r) for r in results])
    print("\n" + "=" * 60)
    print("CHUNKING STRATEGY COMPARISON")
    print("=" * 60)
    print(df.round(2).to_string(index=False))
    print("\nKey Insight: Fixed-512 balances context preservation with retrieval precision.")
    print("Use semantic chunking for domain-specific documents with topic shifts.")
    return df



In [8]:

# ─────────────────────────────────────────────────────────────────────────────
# 3. EMBEDDING & VECTOR STORE SETUP
# ─────────────────────────────────────────────────────────────────────────────
class EmbeddingEngine:
    """
    QUESTION 3 (Embeddings):
    Compare embedding models for airline domain:
      - all-MiniLM-L6-v2: Fast, general purpose (384 dimensions)
      - all-mpnet-base-v2: Higher quality, slower (768 dimensions)

    Domain-specific embeddings outperform general embeddings on
    technical documents. Show this with retrieval metrics.
    """
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        if SENTENCE_TRANSFORMERS:
            self.model = SentenceTransformer(model_name)
            self.dim = self.model.get_sentence_embedding_dimension()
        else:
            self.model = None
            self.dim = 384
        self.model_name = model_name
        logger.info(f"Embedding model: {model_name} (dim={self.dim})")

    def encode(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        if self.model:
            return self.model.encode(texts, batch_size=batch_size,
                                      show_progress_bar=False, normalize_embeddings=True)
        else:
            # Random embeddings for demonstration if sentence_transformers unavailable
            np.random.seed(42)
            return np.random.randn(len(texts), self.dim).astype(np.float32)

    def similarity(self, query_emb: np.ndarray, doc_embs: np.ndarray) -> np.ndarray:
        """Cosine similarity (works because we normalize embeddings)."""
        return np.dot(doc_embs, query_emb)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. HYBRID SEARCH (Dense + Sparse BM25)
# ─────────────────────────────────────────────────────────────────────────────
class HybridSearchEngine:
    """
    QUESTION 4 (Hybrid Search — what enterprise RAG systems actually use):
    Vector search alone fails for exact keyword queries
    (e.g., 'Rs. 3,500 cancellation fee' → vectors may miss exact amounts).

    BM25 keyword search alone misses semantic similarity
    (e.g., 'baggage fee' vs 'checked luggage charge').

    Hybrid search combines both with a weighted reciprocal rank fusion:
      final_score = α × dense_rank_score + (1-α) × bm25_rank_score

    This is the architecture used by Elastic, Pinecone, and Weaviate.
    """

    def __init__(self, documents: List[Dict],
                 embedding_engine: EmbeddingEngine,
                 dense_weight: float = 0.7):
        self.documents = documents
        self.engine = embedding_engine
        self.dense_weight = dense_weight
        self.bm25_weight = 1 - dense_weight

        # Build BM25 index
        self._build_bm25()

        # Build dense index
        texts = [d["content"] for d in documents]
        self.embeddings = self.engine.encode(texts)
        logger.info(f"Indexed {len(documents)} documents "
                    f"(dim={self.embeddings.shape[1]}, hybrid_α={dense_weight})")

    def _build_bm25(self):
        """Build BM25 index from document texts."""
        try:
            from rank_bm25 import BM25Okapi
            tokenized = [d["content"].lower().split() for d in self.documents]
            self.bm25 = BM25Okapi(tokenized)
            self.bm25_available = True
        except ImportError:
            logger.warning("rank_bm25 not installed. Run: pip install rank-bm25")
            self.bm25_available = False

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Hybrid search: reciprocal rank fusion of dense + BM25 results.
        """
        t0 = time.time()

        # Dense retrieval
        q_emb = self.engine.encode([query])[0]
        dense_scores = self.engine.similarity(q_emb, self.embeddings)
        dense_ranks  = np.argsort(-dense_scores)

        # BM25 retrieval
        if self.bm25_available:
            bm25_scores = self.bm25.get_scores(query.lower().split())
            bm25_ranks  = np.argsort(-bm25_scores)
        else:
            bm25_ranks = dense_ranks  # Fallback

        # Reciprocal Rank Fusion
        k = 60  # RRF constant
        rrf_scores = np.zeros(len(self.documents))
        for rank, idx in enumerate(dense_ranks):
            rrf_scores[idx] += self.dense_weight * (1 / (rank + k))
        for rank, idx in enumerate(bm25_ranks):
            rrf_scores[idx] += self.bm25_weight * (1 / (rank + k))

        top_indices = np.argsort(-rrf_scores)[:top_k]
        latency_ms = (time.time() - t0) * 1000

        results = []
        for idx in top_indices:
            doc = dict(self.documents[idx])
            doc["rrf_score"]   = float(rrf_scores[idx])
            doc["dense_score"] = float(dense_scores[idx])
            doc["rank"]        = int(np.where(top_indices == idx)[0][0]) + 1
            results.append(doc)

        logger.info(f"Hybrid search: '{query[:50]}...' → {len(results)} results in {latency_ms:.1f}ms")
        return results


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. RAG PIPELINE WITH HALLUCINATION DETECTION
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class RAGResponse:
    query: str
    answer: str
    retrieved_chunks: List[Dict]
    faithfulness_score: float
    answer_relevancy_score: float
    context_precision_score: float
    latency_ms: float
    token_cost_usd: float
    hallucination_flag: bool
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class RAGPipeline:
    """
    QUESTION 5 (Production RAG Pipeline):
    Build a complete, evaluation-ready RAG pipeline with:
      - Semantic caching (avoid redundant LLM calls)
      - Hallucination detection (NLI-based faithfulness check)
      - Token cost tracking (critical for production budget control)
      - Response logging for feedback loop
    """

    def __init__(self, search_engine: HybridSearchEngine):
        self.search_engine = search_engine
        self.cache: Dict[str, RAGResponse] = {}
        self.response_log: List[RAGResponse] = []
        self.total_tokens = 0
        self.total_cost_usd = 0.0

    def _get_cache_key(self, query: str) -> str:
        return hashlib.md5(query.lower().strip().encode()).hexdigest()

    def _build_prompt(self, query: str, context_chunks: List[Dict]) -> str:
        context = "\n\n".join([
            f"[Source: {c.get('source','unknown')}, Page: {c.get('page','N/A')}]\n{c['content']}"
            for c in context_chunks
        ])
        return f"""You are an airline customer support assistant. Answer the user's question
ONLY using the information in the provided context. If the answer is not in the context,
say "I don't have that information in our policy documents."

CONTEXT:
{context}

QUESTION: {query}

ANSWER (cite the specific policy document and page number in your response):"""

    def query(self, user_query: str, top_k: int = 5,
              use_cache: bool = True) -> RAGResponse:
        """
        Full RAG query with evaluation metrics computed per response.
        """
        t0 = time.time()

        # Check semantic cache
        cache_key = self._get_cache_key(user_query)
        if use_cache and cache_key in self.cache:
            logger.info(f"Cache hit for query: '{user_query[:40]}...'")
            return self.cache[cache_key]

        # Retrieve
        chunks = self.search_engine.search(user_query, top_k=top_k)

        # Generate (simulated — replace with actual LLM call)
        answer, tokens_used, cost = self._call_llm(user_query, chunks)

        # Evaluate (RAGAS-inspired metrics)
        faithfulness  = self._score_faithfulness(answer, chunks)
        relevancy     = self._score_answer_relevancy(user_query, answer)
        ctx_precision = self._score_context_precision(user_query, chunks)
        hallucination = faithfulness < 0.6  # Flag if <60% faithful to context

        latency = (time.time() - t0) * 1000
        self.total_tokens += tokens_used
        self.total_cost_usd += cost

        response = RAGResponse(
            query=user_query,
            answer=answer,
            retrieved_chunks=chunks,
            faithfulness_score=faithfulness,
            answer_relevancy_score=relevancy,
            context_precision_score=ctx_precision,
            latency_ms=latency,
            token_cost_usd=cost,
            hallucination_flag=hallucination,
        )

        self.cache[cache_key] = response
        self.response_log.append(response)

        if hallucination:
            logger.warning(f"⚠ HALLUCINATION DETECTED for query: '{user_query[:50]}'")

        return response

    def _call_llm(self, query: str, chunks: List[Dict]) -> Tuple[str, int, float]:
        """
        Call an LLM API. Replace this with actual OpenAI / Anthropic / Gemini call.
        Returns: (answer, tokens_used, cost_usd)
        """
        prompt = self._build_prompt(query, chunks)

        try:
            import openai
            client = openai.OpenAI()
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=500,
            )
            answer = resp.choices[0].message.content
            tokens = resp.usage.total_tokens
            cost   = tokens * 0.00000015  # gpt-4o-mini pricing
            return answer, tokens, cost
        except Exception:
            # Simulated answer for demonstration
            if chunks:
                top_chunk = chunks[0]["content"]
                answer = (f"Based on our policy documents: {top_chunk[:300]}... "
                          f"[Simulated response — connect your LLM API key]")
            else:
                answer = "I don't have that information in our policy documents."
            return answer, 200, 0.00003

    def _score_faithfulness(self, answer: str, chunks: List[Dict]) -> float:
        """
        Faithfulness: What fraction of the answer's claims are supported by the context?
        Production implementation uses an NLI model (e.g., cross-encoder/nli-MiniLM-L-6-v2).
        Here we use a heuristic: overlap of key terms.
        """
        context = " ".join([c["content"] for c in chunks]).lower()
        answer_words = set(answer.lower().split())
        context_words = set(context.split())
        overlap = len(answer_words & context_words) / (len(answer_words) + 1e-8)
        return min(1.0, overlap * 3)  # Scale up overlap fraction

    def _score_answer_relevancy(self, query: str, answer: str) -> float:
        """
        Answer Relevancy: Does the answer actually address the question?
        Production: embed both query and answer, compute cosine similarity.
        """
        query_words  = set(query.lower().split())
        answer_words = set(answer.lower().split())
        overlap = len(query_words & answer_words) / (len(query_words) + 1e-8)
        return min(1.0, overlap * 5)

    def _score_context_precision(self, query: str, chunks: List[Dict]) -> float:
        """
        Context Precision: What fraction of retrieved chunks are actually relevant?
        Production: use an LLM judge or cross-encoder to score chunk relevance.
        """
        query_words = set(query.lower().split())
        relevant = sum(1 for c in chunks
                       if len(query_words & set(c["content"].lower().split())) > 2)
        return relevant / max(len(chunks), 1)
